**TEMA:** Classificação de Sentimento de Avaliações de Produtos

* **O Problema Real:** Empresas de e-commerce e consumidores precisam entender a opinião sobre produtos. Análises manuais são impraticáveis devido ao volume de dados. Um modelo de aprendizado de máquina pode automatizar esse processo, classificando avaliações como "positivas", "neutras" ou "negativas".

* **O Algoritmo:** Embora o NLP (Processamento de Linguagem Natural) seja uma área à parte, a classificação de sentimentos é um problema que pode ser resolvido com uma **MLP**, tratando o texto como dados tabulares após uma etapa de pré-processamento. Ou, de forma mais avançada, uma **CNN** também pode ser usada para analisar sequências de palavras, mas a MLP é mais que suficiente e mais simples para o objetivo do seu trabalho.

* **O Dataset:** [https://www.kaggle.com/datasets/salahuddinahmedshuvo/ecommerce-consumer-behavior-analysis-data](https://www.kaggle.com/datasets/salahuddinahmedshuvo/ecommerce-consumer-behavior-analysis-data).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib
import re

In [ ]:
PATH = "Ecommerce_Consumer_Behavior_Analysis_Data.csv"
df = pd.read_csv(PATH, low_memory=False)
print("shape:", df.shape)
print("columns:", df.columns.tolist())

# Definindo as colunas
text_col = "Review"           # Alterar e colocar o nome da coluna com o texto das avaliações
rating_col = "Product_Rating" # já presente no seu dataset

# Verificação rápida
if text_col not in df.columns:
    raise ValueError(f"Coluna de texto '{text_col}' não encontrada. Ajuste text_col para a coluna certa.")
if rating_col not in df.columns:
    raise ValueError(f"Coluna de rating '{rating_col}' não encontrada. Ajuste rating_col.")

# Limpeza básica e a criação de rótulos de sentimento a partir da nota
data = df[[text_col, rating_col]].dropna().copy()
data['rating_num'] = pd.to_numeric(data[rating_col], errors='coerce')
data = data.dropna(subset=['rating_num'])
def rating_to_sent(r):
    if r >= 4: return 'positive'
    if r == 3: return 'neutral'
    return 'negative'
data['sentiment'] = data['rating_num'].apply(rating_to_sent)
print(data['sentiment'].value_counts())

MAX_SAMPLES = 20000
if len(data) > MAX_SAMPLES:
    data = data.groupby('sentiment', group_keys=False).apply(
        lambda x: x.sample(min(len(x), MAX_SAMPLES//3), random_state=42)
    ).sample(MAX_SAMPLES, random_state=42)

X = data[text_col].astype(str).values
y = data['sentiment'].values
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("classes:", le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

# TF-IDF + MLP
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1,2), stop_words='english')),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(256,),
        activation='relu',
        solver='adam',
        early_stopping=True,
        learning_rate_init=1e-3,
        max_iter=50,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)

# Avaliação do modelo
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification report:\n", classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

joblib.dump({'pipeline': pipeline, 'label_encoder': le}, "sentiment_mlp.joblib")
print("Modelo salvo em sentiment_mlp.joblib")


FileNotFoundError: [Errno 2] No such file or directory: 'Ecommerce_Consumer_Behavior_Analysis_Data.csv'